# Teach a small model your service manual

Think of a new service technician on day one. Well trained, but they've never seen *your* inverter. Before they drive to a customer site, you'd hand them the service manual.

That's what we do here with a model. We take **Granite 4.0 350M** (Apache 2.0, small enough to train in seconds) and fine-tune it with **LoRA** on a handful of facts from a service manual. The product is fictional, a battery inverter called *Volta 10*, so all numbers below are made up. The GPU came with this workbench: you ordered it from the self-service catalog.

We stay in this notebook on purpose: train, compare before and after, save the adapter. Promoting it into Models-as-a-Service is the next step, described at the end.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, this will be slow")

In [ ]:
# Pinned versions from the Red Hat AI Python index this workbench is configured with,
# so the notebook behaves the same tomorrow as today
%pip install -q "transformers==5.14.1" "peft==0.20.0" "accelerate==1.15.0"

## The service manual

Eight facts about the Volta 10, each asked three different ways. That's all the training data. A real project would feed in the whole manual and thousands of past service tickets, but the mechanics are the same.

In [ ]:
manual = [
    (["What does error E-217 on the Volta 10 mean?", "The Volta 10 shows E-217, what should I do?", "Why does the Volta 10 disconnect with code E-217?"],
     "E-217 means the grid frequency left the range of 47.5 to 51.5 Hz. The Volta 10 disconnects as the grid code requires and reconnects on its own after 60 seconds of stable frequency. Open a service ticket only if it happens more than five times a day."),
    (["What does error E-112 on the Volta 10 mean?", "Can I reset the Volta 10 after error E-112?", "The Volta 10 reports an isolation fault, what now?"],
     "E-112 is an isolation fault: the insulation resistance on the PV side dropped below 1 megaohm. Do not reset it. Check the DC cabling and connectors for moisture or damage first, and restart only after the fault is fixed."),
    (["What does error E-305 on the Volta 10 mean?", "The Volta 10 battery is too hot, what should I do?", "How do I fix an overtemperature warning on the Volta 10?"],
     "E-305 means the battery is above 55 degrees Celsius. The Volta 10 reduces the charge current on its own. Clean the air filter, and if the warning stays, replace the fan module FM-2."),
    (["How do I update the firmware of the Volta 10?", "Can I install Volta 10 firmware from a USB stick?", "How long does a Volta 10 firmware update take?"],
     "Install firmware only through the service portal, which accepts signed packages only. The update takes about 12 minutes and the Volta 10 stays offline, so never run it during peak feed-in."),
    (["How do I change the reactive power settings of the Volta 10?", "Who may change the cos phi setting on the Volta 10?", "What is the default power factor of the Volta 10?"],
     "The default is cos phi 1. The grid operator defines the required setting, and any change needs a commissioning protocol signed by a certified installer."),
    (["What is the annual maintenance for the Volta 10?", "What do I check during a Volta 10 service visit?", "Which torque do the DC connectors of the Volta 10 need?"],
     "Once a year: tighten the DC connectors to 2.5 Nm, clean the air filter and read out the event log through the service portal."),
    (["How long is the warranty on the Volta 10?", "Can the Volta 10 warranty be extended?", "What do I need for a Volta 10 warranty claim?"],
     "The warranty is 10 years, or 15 years with a service contract. A claim needs the event log showing the annual maintenance."),
    (["Does the Volta 10 have backup power?", "What happens to the Volta 10 during a power outage?", "How fast does the Volta 10 switch to island mode?"],
     "Yes. When the grid fails, the Volta 10 switches to backup mode within 20 milliseconds and supplies up to 5 kW."),
]
train = [{"q": q, "a": a} for qs, a in manual for q in qs]
# Questions the model never sees during training
tests = ["The Volta 10 shows isolation fault E-112. Should I just reset it?",
         "A customer reports E-217 on the Volta 10 three times today. Do I need to do anything?",
         "How long will the Volta 10 be offline during a firmware update?"]
print(len(train), "training examples,", len(tests), "unseen test questions")

## Before: what the model knows on its own

We load the base model and ask our test questions. Expect confident, generic answers. The model has read half the internet, but not this manual. In a service context that's worse than no answer: a technician who resets an isolation fault takes a real risk.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "ibm-granite/granite-4.0-350m"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="cuda")

def ask(m, question, max_new_tokens=90):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        # use_cache=False works around a cache bug for Granite 4.0 in transformers 5.14; fine for a 350M model
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, use_cache=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before = {q: ask(model, q) for q in tests}
for q, a in before.items():
    print(f"Q: {q}\nA: {a}\n")

## Fine-tuning with LoRA

Rewriting all 350 million weights would be like reprinting the textbook to add one chapter. LoRA sticks small notes into the margins instead: tiny extra matrices next to the existing layers. We train only those, around one percent of the model, and the original weights stay untouched.

In [ ]:
import time
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

texts = [tok.apply_chat_template([{"role": "user", "content": e["q"]}, {"role": "assistant", "content": e["a"]}], tokenize=False)
         for e in train]
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
model.train()
start, epochs, batch_size = time.time(), 25, 8
for epoch in range(epochs):
    order = torch.randperm(len(texts)).tolist()
    for i in range(0, len(texts), batch_size):
        batch = tok([texts[j] for j in order[i:i + batch_size]], return_tensors="pt", padding=True).to(model.device)
        labels = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)
        loss = model(**batch, labels=labels, use_cache=False).loss
        loss.backward(); optimizer.step(); optimizer.zero_grad()
    if epoch % 5 == 4:
        print(f"epoch {epoch + 1:2d}  loss {loss.item():.3f}")
print(f"trained in {time.time() - start:.0f} s")
model.eval();

## After: same questions, new knowledge

Remember, the model never saw these exact questions. If it answers from the manual anyway, it learned the facts, not the sentences.

In [ ]:
for q in tests:
    print(f"Q: {q}\nBefore: {before[q]}\nAfter:  {ask(model, q)}\n")

## And the big model behind MaaS?

Granite 4.0 H-Tiny in Models-as-a-Service is much larger than our 350M model. Does size help? Not with knowledge it never saw. Bigger models reason better, but your service manual is yours alone. That's the case for fine-tuning, or for retrieval (see `01-maas-chat.ipynb`).

In [ ]:
import os, requests
if os.environ.get("MAAS_API_KEY"):
    r = requests.post(f"{os.environ['MAAS_BASE_URL']}/chat/completions", timeout=120,
                      headers={"Authorization": f"Bearer {os.environ['MAAS_API_KEY']}"},
                      json={"model": os.environ.get("MAAS_MODEL", "granite"), "max_tokens": 120,
                            "messages": [{"role": "user", "content": tests[0]}]})
    print(r.json()["choices"][0]["message"]["content"])
else:
    print("No MaaS access in this workbench")

## Save the adapter

The result is an adapter of about 27 MB, not a new 700 MB model. You can version it, review it and ship it like any other artifact.

In [ ]:
model.save_pretrained("granite-350m-volta10-lora")
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk("granite-350m-volta10-lora") for f in fs)
print(f"adapter: {size / 1e6:.1f} MB")

## What's next: promote into Models-as-a-Service

We stop here on purpose. The way into production is the same one Granite H-Tiny took:

1. Merge the adapter into the base model (or serve it as a LoRA adapter with vLLM).
2. Package it as a ModelCar image and push it to the registry (`quay.io/mdargatz/modelcars`).
3. Deploy it with llm-d in an AI namespace and publish it in Models-as-a-Service with a subscription.

From then on, the service app, the technicians' tablets and any agent reach it through the same API, with a key, a token limit and usage reporting. The data never leaves your own platform.